# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent")

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    logger.info(f"Calculator tool called with expression: '{expression}'")
    try:
        # Only allow safe characters to reduce risk from eval()
        allowed_chars = set("0123456789+-*/(). ")
        if not expression or not set(expression) <= allowed_chars:
            raise ValueError("Expression contains invalid characters")
        return str(eval(expression))
    except Exception as e:
        logger.error(f"Calculator error: {e}")
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    logger.info(f"Keyword tool called with text: '{text}'")
    try:
        words = text.split()
        keywords = list(set([w.lower().strip('.,!?;:"\'') for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception as e:
        logger.error(f"Keyword extractor error: {e}")
        return []

In [3]:
# 🛠️ TOOL 3 (BONUS): Word Counter

def count_words(text: str) -> int:
    """Count the number of words in a piece of text."""
    logger.info(f"Word counter tool called with text: '{text}'")
    try:
        return len(text.split())
    except Exception as e:
        logger.error(f"Word counter error: {e}")
        return 0

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [4]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

def agent(query: str):
    """
    Routes a user query to the correct tool based on intent, and returns a
    structured JSON-style response: {"type": ..., "result": ...}
    """
    logger.info(f"Agent received query: '{query}'")

    if not query or not query.strip():
        return {"type": "error", "result": "Empty query received."}

    query_lower = query.lower()

    try:
        # ---- Route 1: Math / Calculation ----
        if "calculate" in query_lower:
            expression = query_lower.split("calculate", 1)[1].strip()
            if not expression:
                logger.warning("No expression found after 'calculate'")
                return {"type": "error", "result": "No expression found to calculate."}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}

            return {"type": "calculation", "result": calc_result}

        # ---- Route 2: Keyword Extraction ----
        elif "keyword" in query_lower:
            # Try to isolate the text to extract keywords from (text after 'from', if present)
            if " from " in query_lower:
                text = query.split("from", 1)[1].strip()
            else:
                text = query

            if not text:
                return {"type": "error", "result": "No text found to extract keywords from."}

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # ---- Route 3 (BONUS): Word Count ----
        elif "word count" in query_lower or "count words" in query_lower:
            text = query_lower.replace("word count", "").replace("count words", "").replace("of", "").strip()
            if not text:
                return {"type": "error", "result": "No text found to count words in."}
            return {"type": "word_count", "result": count_words(text)}

        # ---- Route 4: General / Fallback ----
        else:
            response = (
                f"I received your query: '{query}'. "
                "I can help with calculations (say 'calculate ...'), "
                "keyword extraction (say 'keywords from ...'), "
                "or word counts (say 'word count of ...')."
            )
            return {"type": "general", "result": response}

    except Exception as e:
        logger.error(f"Agent encountered an unexpected error: {e}")
        return {"type": "error", "result": f"Something went wrong: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [5]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate",                     # edge case: no expression
    "Word count of the quick brown fox jumps over the lazy dog",  # bonus route
    "Calculate 10 / 0"                # edge case: math error
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-07-19 03:00:57,318 [INFO] Agent received query: 'Calculate 20 + 5'
2026-07-19 03:00:57,319 [INFO] Calculator tool called with expression: '20 + 5'
2026-07-19 03:00:57,319 [INFO] Agent received query: 'Extract keywords from Artificial Intelligence is transforming industries'
2026-07-19 03:00:57,320 [INFO] Keyword tool called with text: 'Artificial Intelligence is transforming industries'
2026-07-19 03:00:57,320 [INFO] Agent received query: 'What is machine learning?'
2026-07-19 03:00:57,320 [INFO] Agent received query: 'Calculate'
2026-07-19 03:00:57,321 [WARNING] No expression found after 'calculate'
2026-07-19 03:00:57,321 [INFO] Agent received query: 'Word count of the quick brown fox jumps over the lazy dog'
2026-07-19 03:00:57,321 [INFO] Word counter tool called with text: 'the quick brown fox jumps over the lazy dog'
2026-07-19 03:00:57,322 [INFO] Agent received query: 'Calculate 10 / 0'
2026-07-19 03:00:57,322 [INFO] Calculator tool called with expression: '10 / 0'
2026-07-1

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'industries', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. I can help with calculations (say 'calculate ...'), keyword extraction (say 'keywords from ...'), or word counts (say 'word count of ...')."}
--------------------------------------------------
Query: Calculate
Response: {'type': 'error', 'result': 'No expression found to calculate.'}
--------------------------------------------------
Query: Word count of the quick brown fox jumps over the lazy dog
Response: {'type': 'word_count', 'result': 9}
--------------------------------------------------
Query: Calculate 

In [6]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

2026-07-19 03:00:57,373 [INFO] Agent received query: ''


Response: {'type': 'error', 'result': 'Empty query received.'}


2026-07-19 03:02:37,252 [INFO] Agent received query: ''


Response: {'type': 'error', 'result': 'Empty query received.'}


2026-07-19 03:02:40,046 [INFO] Agent received query: ''


...

Response: {'type': 'error', 'result': 'Empty query received.'}


2026-07-19 03:02:42,704 [INFO] Agent received query: ''


Response: {'type': 'error', 'result': 'Empty query received.'}


2026-07-19 03:02:44,393 [INFO] Agent received query: ''


Response: {'type': 'error', 'result': 'Empty query received.'}
